# **Apriori Analysis on Pakistan Traffic Accidents Dataset**

This repository implements the **Apriori Analysis** on a dataset of traffic accidents in Pakistan. The dataset contains only **64 rows** of data in a CSV file, which limits the depth of analysis. Due to the small dataset size, the process generates candidate itemsets only up to **Candidate 2 (C2)**.

The analysis identifies frequent patterns and associations between attributes like **Fatal Accidents**, **Non-Fatal Accidents**, **Number of Vehicles Involved**, and more. By applying thresholds like minimum support, the frequent itemsets provide insights into recurring patterns in traffic accidents.

This project serves as a practical example of using the Apriori algorithm on smaller datasets, demonstrating how to handle and analyze limited data effectively.

# Step 1: Generate C1 (Candidate Set with Support Counts)

## Objective:
The goal of this step is to generate **C1**, a candidate set that contains each individual item from the dataset along with its support count. This is the foundation for further frequent itemset mining using the Apriori algorithm.

---

## How the Code Works:

### **1. Load Dataset**
- The dataset (`pak-traffic-accidents-annual.csv`) is loaded using `pandas.read_csv`.
- It contains attributes like `Total number of accidents`, `Fatal Accidents`, etc.

### **2. Apply Threshold-Based Categorization**
- Each numerical attribute is categorized into binary values (e.g., `> threshold` or `<= threshold`).
- This helps in creating transactions suitable for mining frequent itemsets.

#### **Example**:
For a row:

- Total number of accidents = 6000 Fatal Accidents = 3000

- With thresholds:

- Total number of accidents > 5000 Fatal Accidents > 2000

- The resulting transaction will contain:

- "Total number of accidents > 5000", "Fatal Accidents > 2000"


### **3. Generate Transactions**
- Each row in the dataset is converted into a transaction (a list of categorized attributes).

### **4. Generate C1**
- For each transaction, the code counts the number of times each item appears across all transactions.
- The result is a dictionary (`C1`) where:
  - **Key**: Item (e.g., `"Total number of accidents > 5000"`)
  - **Value**: Support count (number of transactions containing that item).

### **5. Display Results**
- The candidate set (`C1`) is converted into a tabular format using `pandas.DataFrame`.
- It is displayed as a table for easy visualization.

---

## Output:
- A table displaying each item and its support count.

In [63]:
# Step-1: Adjust the code to read data from a CSV file for C1 generation

import pandas as pd

# Load the dataset
data = pd.read_csv("Data/pak-traffic-accidents-annual.csv")

# Define thresholds for categorization (example thresholds, adjust as needed)
thresholds = {
    "Total number of accidents": 5000,
    "Fatal Accidents": 2000,
    "Non-Fatal Accidents": 3000,
    "Killed": 3000,
    "Injured": 8000,
    "Total number of vehicles involved": 5000
}

# Convert data into binary transactions based on thresholds
def categorize(row, thresholds):
    transaction = []
    for col, threshold in thresholds.items():
        if row[col] > threshold:
            transaction.append(f"{col} > {threshold}")
        else:
            transaction.append(f"{col} <= {threshold}")
    return transaction

# Create transactions
transactions = data.apply(lambda row: categorize(row, thresholds), axis=1).tolist()

# Generate C1: Count support for each individual item
def generate_c1(transactions):
    """
    Generate C1, a candidate set of individual items with their support counts.

    Parameters:
        transactions (list): List of transactions, where each transaction is a set of items.

    Returns:
        dict: Dictionary of items and their support counts.
    """
    c1 = {}
    for transaction in transactions:
        for item in transaction:
            c1[item] = c1.get(item, 0) + 1
    return c1

# Generate C1
c1 = generate_c1(transactions)

# Convert C1 to a DataFrame for better visualization
c1_df = pd.DataFrame({
    "Item": list(c1.keys()),
    "Support Count": list(c1.values())
})

# Display C1
from IPython.display import display, HTML
print("C1 (Candidate Set with Support Counts):")
display(HTML(c1_df.to_html(index=False)))


C1 (Candidate Set with Support Counts):


Item,Support Count
Total number of accidents > 5000,15
Fatal Accidents > 2000,18
Non-Fatal Accidents > 3000,15
Killed > 3000,15
Injured > 8000,11
Total number of vehicles involved > 5000,19
Non-Fatal Accidents <= 3000,47
Killed <= 3000,47
Injured <= 8000,51
Total number of accidents <= 5000,47


## Step II: Generate L1 (Filtered Itemset with Support Counts ≥ Minimum Support)

### Objective:
Filter the candidate set (`C1`) by removing items with support counts below a specified minimum support threshold. The resulting filtered itemset is called `L1`.

---

### **How the Code Works**

1. **Input**:
   - Takes the `C1` candidate set generated in Step I, which contains items and their support counts.
   - A predefined minimum support threshold (`min_support`) is set. For example:
     ```python
     min_support = 2
     ```

2. **Filtering Logic**:
   - Each item in `C1` is compared against the `min_support` threshold.
   - If an item's support count is greater than or equal to `min_support`, it is retained in the filtered set `L1`.
   - Items that do not meet this condition are removed.

3. **Output**:
   - The filtered set `L1`, containing items and their support counts, is stored as a dictionary.
   - This dictionary is converted to a `pandas.DataFrame` for tabular display.

4. **Visualization**:
   - The resulting `L1` is displayed as an HTML table for better readability in a Jupyter Notebook environment.

---

In [84]:
# Step (II): Filter C1 using minimum support threshold to generate L1

# Define minimum support threshold
min_support = 17

# Generate L1 by filtering items in C1 with support count >= min_support
l1 = {item: support for item, support in c1.items() if support >= min_support}

# Convert L1 to a DataFrame for better visualization
l1_df = pd.DataFrame({
    "Item": list(l1.keys()),
    "Support Count": list(l1.values())
})

# Display L1
from IPython.display import display, HTML
display(HTML(l1_df.to_html(index=False)))


Item,Support Count
Fatal Accidents > 2000,18
Total number of vehicles involved > 5000,19
Non-Fatal Accidents <= 3000,47
Killed <= 3000,47
Injured <= 8000,51
Total number of accidents <= 5000,47
Total number of vehicles involved <= 5000,43
Fatal Accidents <= 2000,44


# Generating Candidate 2-itemsets (C2) and Support Counts

## Objective:
To generate all possible 2-itemsets (**C2**) from frequent 1-itemsets (**L1**) and calculate their support counts based on transaction data.

---

## How the Code Works:

### **1. Generate Candidate 2-itemsets (C2)**
The function `generate_c2`:
- Takes **L1** (frequent 1-itemsets) as input.
- Creates all possible pairs of items from **L1** as candidate 2-itemsets.
- Ensures each itemset is a valid 2-itemset.

#### Example:
If **L1** contains:
"Item A": 5, "Item B": 7, "Item C": 6 

The resulting **C2** will include:
("Item A", "Item B"), ("Item A", "Item C"), ("Item B", "Item C")

### **2. Count Support for C2**
The function `count_support_c2`:
- Iterates through transactions to count how many times each candidate 2-itemset in **C2** appears.
- Outputs a dictionary where:
  - **Key**: Candidate 2-itemset.
  - **Value**: Support count (number of transactions containing the itemset).

#### Example:
For transactions:
{"Item A", "Item B"}, {"Item A", "Item C"}, {"Item B", "Item C"}

And **C2**:
("Item A", "Item B"), ("Item A", "Item C"), ("Item B", "Item C")

The resulting support counts might be:
("Item A", "Item B"): 1, ("Item A", "Item C"): 1, ("Item B", "Item C"): 1


---

### **3. Include All Candidates (Including Zero Support)**
Unlike filtering candidates based on frequent subsets:
- This code includes all candidate 2-itemsets from **C2**.
- Support counts are calculated for all itemsets, even if their support is zero.

---

### **4. Display Results**
- Converts the support counts into a DataFrame for easier visualization.
- Displays the results as an HTML table.

#### Example Output:
| Itemset                  | Support Count |
|--------------------------|---------------|
| Item A & Item B          | 1             |
| Item A & Item C          | 1             |
| Item B & Item C          | 0             |

---

## Purpose:
This step ensures that all candidate 2-itemsets are generated and evaluated, providing a complete view of their support counts, regardless of whether they meet a specific support threshold.

---

## Use Cases:
- Preparing for further pruning of **C2** to generate **L2** based on a minimum support threshold.
- Debugging and analyzing the distribution of 2-itemsets in the dataset.


In [112]:
# Generate Candidate 2-itemsets (C2) using L1
def generate_c2(l1_items):
    c2 = []
    l1_list = list(l1_items.keys())
    for i in range(len(l1_list)):
        for j in range(i + 1, len(l1_list)):
            candidate = tuple(sorted(set([l1_list[i], l1_list[j]])))  # Create 2-itemsets
            if len(candidate) == 2:  # Ensure it is a 2-itemset
                c2.append(candidate)
    return c2

# Check if all subsets of an itemset are frequent
def is_frequent(itemset, l1):
    subsets = combinations(itemset, len(itemset) - 1)
    return all(tuple(sorted(subset)) in l1 for subset in subsets)

# Generate C2 from L1
c2 = generate_c2(l1)

# Filter C2 to retain all candidates regardless of frequent subsets
filtered_c2 = c2  # Include all candidates

# Count support for C2 itemsets in the dataset
def count_support_c2(c2, transactions):
    support_counts = {itemset: 0 for itemset in c2}
    for transaction in transactions:
        for itemset in c2:
            if set(itemset).issubset(transaction):
                support_counts[itemset] += 1
    return support_counts

# Count support for C2
c2_support = count_support_c2(filtered_c2, transactions)

# Convert C2 with support counts to DataFrame for display
import pandas as pd
from IPython.display import display, HTML

c2_df = pd.DataFrame({
    "Itemset": [' & '.join(itemset) for itemset in c2_support.keys()],
    "Support Count": list(c2_support.values())
})

# Display C2
print("C2 (Candidate 2-itemsets with Support Counts):")
display(HTML(c2_df.to_html(index=False)))

C2 (Candidate 2-itemsets with Support Counts):


Itemset,Support Count
Fatal Accidents > 2000 & Total number of vehicles involved > 5000,0
Fatal Accidents > 2000 & Non-Fatal Accidents <= 3000,0
Fatal Accidents > 2000 & Killed <= 3000,0
Fatal Accidents > 2000 & Injured <= 8000,0
Fatal Accidents > 2000 & Total number of accidents <= 5000,0
Fatal Accidents > 2000 & Total number of vehicles involved <= 5000,0
Fatal Accidents <= 2000 & Fatal Accidents > 2000,0
Non-Fatal Accidents <= 3000 & Total number of vehicles involved > 5000,0
Killed <= 3000 & Total number of vehicles involved > 5000,0
Injured <= 8000 & Total number of vehicles involved > 5000,0


# Generating L2 (Frequent 2-itemsets) from C2

## Objective:
To filter candidate 2-itemsets (**C2**) based on a minimum support threshold to generate **L2**, the set of frequent 2-itemsets.

---

## How the Code Works:

### **1. Define Minimum Support Threshold**
- A minimum support threshold (`min_support`) is defined, below which itemsets are considered infrequent and are removed.

#### Example:
If `min_support = 2`, any 2-itemset with a support count less than 2 will be excluded.

---

### **2. Filter C2 to Generate L2**
The code:
- Iterates through the support counts of all candidates in **C2**.
- Retains only those 2-itemsets where the support count is greater than or equal to `min_support`.
- Creates **L2**, a dictionary of frequent 2-itemsets and their support counts.

#### Example:
For **C2**:
("Item A", "Item B"): 1, ("Item A", "Item C"): 2, ("Item B", "Item C"): 3

With `min_support = 2`, the resulting **L2** will be:
("Item A", "Item C"): 2, ("Item B", "Item C"): 3

---

### **3. Convert to DataFrame**
The filtered **L2** is converted into a pandas `DataFrame` for better visualization, with columns:
- **Itemset**: The frequent 2-itemset (e.g., `"Item A & Item B"`).
- **Support Count**: The number of transactions containing the itemset.

---

### **4. Display Results**
The results are displayed in an HTML table if **L2** is not empty:
- Example Table:
| Itemset                  | Support Count |
|--------------------------|---------------|
| Item A & Item C          | 2             |
| Item B & Item C          | 3             |

- If **L2** is empty, a message is printed:
"L2 is empty. No 2-itemsets met the minimum support threshold."

---

## Purpose:
- To identify frequent 2-itemsets that meet the minimum support threshold.
- Prepare the dataset for further steps in the Apriori algorithm, such as generating larger frequent itemsets or association rules.

In [109]:
# Filter C2 to generate L2 based on minimum support threshold
min_support = 1 # Set minimum support threshold

# Generate L2 by filtering C2 candidates with support >= min_support
l2 = {itemset: support for itemset, support in c2_support.items() if support >= min_support}

# Convert L2 to DataFrame for better visualization
l2_df = pd.DataFrame({
    "Itemset": [' & '.join(itemset) for itemset in l2.keys()],
    "Support Count": list(l2.values())
})

# Display L2
if not l2_df.empty:
    print("L2 (Frequent 2-itemsets with Support Counts):")
    from IPython.display import display, HTML
    display(HTML(l2_df.to_html(index=False)))
else:
    print("L2 is empty. No 2-itemsets met the minimum support threshold.")


L2 (Frequent 2-itemsets with Support Counts):


Itemset,Support Count
Killed <= 3000 & Non-Fatal Accidents <= 3000,2
Injured <= 8000 & Non-Fatal Accidents <= 3000,1
Non-Fatal Accidents <= 3000 & Total number of accidents <= 5000,1
Injured <= 8000 & Killed <= 3000,2
Killed <= 3000 & Total number of accidents <= 5000,2
Injured <= 8000 & Total number of accidents <= 5000,1
Injured <= 8000 & Total number of vehicles involved <= 5000,1
Fatal Accidents <= 2000 & Injured <= 8000,1
Fatal Accidents <= 2000 & Total number of vehicles involved <= 5000,1


# **Insights from Filtered L2 Output (Threshold = 1)**

After applying a support threshold of **1**, the filtered **L2** contains itemsets with a support count of at least **1**. Here's an analysis:

---

## **1. Key Observations**

| **Itemset**                                                                | **Support Count** |
|----------------------------------------------------------------------------|-------------------|
| Killed <= 3000 & Non-Fatal Accidents <= 3000                               | 2                 |
| Injured <= 8000 & Non-Fatal Accidents <= 3000                              | 1                 |
| Non-Fatal Accidents <= 3000 & Total number of accidents <= 5000            | 1                 |
| Injured <= 8000 & Killed <= 3000                                           | 2                 |
| Killed <= 3000 & Total number of accidents <= 5000                         | 2                 |
| Injured <= 8000 & Total number of accidents <= 5000                        | 1                 |
| Injured <= 8000 & Total number of vehicles involved <= 5000                | 1                 |
| Fatal Accidents <= 2000 & Injured <= 8000                                  | 1                 |
| Fatal Accidents <= 2000 & Total number of vehicles involved <= 5000        | 1                 |

---

## **2. Observations and Insights**

### **Frequent Itemsets with Support > 1**
- **"Killed <= 3000 & Non-Fatal Accidents <= 3000" (Support = 2)**:
  - Indicates a recurring pattern of lower fatalities and non-fatal accidents.
- **"Injured <= 8000 & Killed <= 3000" (Support = 2)**:
  - Suggests that low fatalities are often associated with fewer injuries.
- **"Killed <= 3000 & Total number of accidents <= 5000" (Support = 2)**:
  - Shows a common trend of lower fatalities occurring with fewer overall accidents.

### **Less Frequent Patterns (Support = 1)**
- Patterns like **"Injured <= 8000 & Total number of vehicles involved <= 5000"** or **"Fatal Accidents <= 2000 & Injured <= 8000"** occur infrequently, but their presence indicates potential one-off or unique scenarios in the data.

### **Dominance of Low Fatalities and Non-Fatal Accidents**
- Most frequent itemsets include **"Killed <= 3000"**, **"Non-Fatal Accidents <= 3000"**, and **"Injured <= 8000"**, highlighting the dataset's skew toward lower severity incidents.

---

## **3. Patterns in Co-Occurrences**
### **Fatal Accidents**
- Rarely co-occur with other attributes, as seen in **"Fatal Accidents <= 2000"** patterns.
- Lower fatalities tend to pair more frequently with lower injuries or fewer accidents.

### **Injuries and Fatalities**
- **"Injured <= 8000 & Killed <= 3000"** is the most frequent pair, emphasizing the correlation between fewer injuries and lower fatalities.

### **Vehicle Involvement**
- Itemsets involving **"Total number of vehicles involved <= 5000"** appear less frequently, suggesting limited data on high vehicle involvement in accidents.
